# Fish Farming / Model Training / Image Spliter

⛳ The labeling for the images was done using: https://labelstud.io/
It was used the feature for **Object Detection**


💡When exporting the masks from Labelstudio as .png images, the masks names may be rewritten in the numerical order that they were downloaded / labeled.

Its important to keep that order because in the next step, they will be paired with their original images.

For example:

sentinel2_17.png -> task-1-xxx.png (Mask)

sentinel2_356.png -> task-2-xxx.png (Mask)

sentinel2_1024.png -> task-3-xxx.png (Mask)

# Variables Configuration

---


In [ ]:
# Input folders
mask_folder = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/png_masks'
npy_folder = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_npy'

# Output folders
output_images_folder = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_images'
output_masks_folder = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_masks'

#Folder where to save the split data
output_base = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/dataset/rgb/'

#Data percentaje to be split
split_ratio = {'train': 0.7, 'val': 0.2, 'test': 0.1}


# Organizing file names

---


In [ ]:
import os
import re
import numpy as np
import shutil

os.makedirs(output_images_folder, exist_ok=True)
os.makedirs(output_masks_folder, exist_ok=True)

# Helper: extract task number and npy index
def extract_task_number(name):
    match = re.search(r'task-(\d+)', name)
    return int(match.group(1)) if match else None

def extract_index_number(name):
    match = re.search(r'sentinel2_(\d+)', name)
    return int(match.group(1)) if match else None

# Collect and sort mask files by task number
mask_files = [f for f in os.listdir(mask_folder) if f.endswith('.png')]
mask_files_sorted = sorted(mask_files, key=extract_task_number)

# Collect and sort npy files by index number
npy_files = [f for f in os.listdir(npy_folder) if f.endswith('.npy')]
npy_files_sorted = sorted(npy_files, key=extract_index_number)

# Sanity check
if len(mask_files_sorted) != len(npy_files_sorted):
    raise ValueError("Mismatch between number of masks and .npy files")

# Mapping and renaming
for i, (mask_file, npy_file) in enumerate(zip(mask_files_sorted, npy_files_sorted), start=1):
    sample_id = f"sample_{i:03d}"
    new_mask_path = os.path.join(output_masks_folder, f"{sample_id}.png")
    new_image_path = os.path.join(output_images_folder, f"{sample_id}.npy")

    shutil.copy(os.path.join(mask_folder, mask_file), new_mask_path)
    shutil.copy(os.path.join(npy_folder, npy_file), new_image_path)

    print(f"✔️ {mask_file} + {npy_file} → {sample_id}")

print("Done! Paired and saved.")

✔️ task-1-annotation-1-by-1-tag-Cage-0.png + sentinel2_17.npy → sample_001
✔️ task-2-annotation-2-by-1-tag-Cage-0.png + sentinel2_27.npy → sample_002
✔️ task-3-annotation-51-by-1-tag-Cage-0.png + sentinel2_28.npy → sample_003
✔️ task-4-annotation-4-by-1-tag-Cage-0.png + sentinel2_43.npy → sample_004
✔️ task-5-annotation-5-by-1-tag-Cage-0.png + sentinel2_173.npy → sample_005
✔️ task-6-annotation-6-by-1-tag-Cage-0.png + sentinel2_182.npy → sample_006
✔️ task-7-annotation-7-by-1-tag-Cage-0.png + sentinel2_211.npy → sample_007
✔️ task-8-annotation-8-by-1-tag-Cage-0.png + sentinel2_279.npy → sample_008
✔️ task-9-annotation-9-by-1-tag-Cage-0.png + sentinel2_294.npy → sample_009
✔️ task-10-annotation-10-by-1-tag-Cage-0.png + sentinel2_357.npy → sample_010
✔️ task-11-annotation-11-by-1-tag-Cage-0.png + sentinel2_360.npy → sample_011
✔️ task-12-annotation-12-by-1-tag-Cage-0.png + sentinel2_368.npy → sample_012
✔️ task-13-annotation-13-by-1-tag-Cage-0.png + sentinel2_415.npy → sample_013
✔️ task

# Splitting Data First Time
---



In [ ]:
import os
import shutil
import random

# Config
input_images_dir = output_images_folder
input_masks_dir = output_masks_folder
random_seed = 42

# Create output folders
for split in split_ratio:
    os.makedirs(os.path.join(output_base, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(output_base, split, 'masks'), exist_ok=True)

# Get sample base names (sample_001, sample_002, ...)
samples = [f[:-4] for f in os.listdir(input_images_dir) if f.endswith('.npy')]
samples.sort()
random.seed(random_seed)
random.shuffle(samples)

# Compute split sizes
total = len(samples)
train_end = int(split_ratio['train'] * total)
val_end = train_end + int(split_ratio['val'] * total)

# Perform splits
splits = {
    'train': samples[:train_end],
    'val': samples[train_end:val_end],
    'test': samples[val_end:]
}

# Copy files
for split, sample_list in splits.items():
    for sample in sample_list:
        shutil.copy(os.path.join(input_images_dir, f"{sample}.npy"),
                    os.path.join(output_base, split, 'images', f"{sample}.npy"))
        shutil.copy(os.path.join(input_masks_dir, f"{sample}.png"),
                    os.path.join(output_base, split, 'masks', f"{sample}.png"))

    print(f" {split}: {len(sample_list)} samples")

print(" Dataset successfully split!")

 train: 35 samples
 val: 10 samples
 test: 5 samples
 Dataset successfully split!


# (Optional) Split data based on a previous split

---


This step is aimed to be used for comparison purposes, since we had to contrast between diferent image formats but keeping the original distribution made for RGB

In [ ]:
import os
import shutil

# Folder A: base structure & names from the first splitted data, to copy their distribution
base_folder = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_trainning/data/dataset/'

# Source flat folders, containing all dataset images and masks
source_images = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_images'
source_masks = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/processed/sentinel2_masks'


# Destination folder of the new split data
dest_folder = '/content/drive/MyDrive/RPTU/SM_25 25/AAI_Project/model_training/data/datasetB/rgb'

for root, _, files in os.walk(base_folder):
    rel_path = os.path.relpath(root, base_folder)   # keeps train/val/test/images/masks
    dest_path = os.path.join(dest_folder, rel_path)

    os.makedirs(dest_path, exist_ok=True)

    for file in files:
        base_name, _ = os.path.splitext(file)

        if 'images' in rel_path:  # copy from image source
            src_folder = source_images
        elif 'masks' in rel_path:  # copy from mask source
            src_folder = source_masks
        else:
            continue  # skip any unrelated folders

        # Look for file in the relevant source folder
        found = False
        for src_file in os.listdir(src_folder):
            if os.path.splitext(src_file)[0] == base_name:
                shutil.copy(
                    os.path.join(src_folder, src_file),
                    os.path.join(dest_path, src_file)
                )
                print(f"[COPIED] {src_file} → {dest_path}")
                found = True
                break

        if not found:
            print(f"[WARNING] {base_name} not found in {src_folder}")